# TSTL Phase R1 — Colab (GPU)

論文再現の **GPU 部分** を Colab で実行。

- Runtime: **GPU** (T4 16GB)
- チェックポイント: Google Drive (`/content/drive/MyDrive/TSTL/r1`)
- ローカル PC では pytest のみ（CPU）

In [ ]:
# Clone or upload the repo, then:
# %cd /content/NSNandTSTL/TSTL

!pip install -q -r requirements-r.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

ROOT = Path('/content/NSNandTSTL/TSTL')  # adjust after clone
sys.path.insert(0, str(ROOT / 'src'))
CKPT_ROOT = Path('/content/drive/MyDrive/TSTL/r1')
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llm_freeze import freeze_all_except_layers, num_transformer_layers
from llm_grpo import GrpoRunConfig, run_grpo_train
from llm_eval import accuracy_score
from llm_profile import default_r1_out_dir, profile_layers_from_scores, resume_layer_scan

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
TRAIN_N = 256
EVAL_N = 64
STEPS = 200
LR = 1e-5
SEED = 42

In [ ]:
def load_gsm8k_prompts(n_train: int, n_eval: int):
    ds = load_dataset('gsm8k', 'main')
    train = ds['train'].select(range(n_train))
    test = ds['test'].select(range(n_eval))
    return train, test


def to_grpo_rows(split, tokenizer):
    rows = []
    for ex in split:
        prompt = f"Question: {ex['question']}\nAnswer:"
        rows.append({"prompt": prompt, "answer": ex['answer']})
    return rows


train_split, eval_split = load_gsm8k_prompts(TRAIN_N, EVAL_N)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
grpo_train = to_grpo_rows(train_split, tokenizer)

In [ ]:
def eval_model(model, split, tokenizer, max_new_tokens=64):
    preds, gold = [], []
    model.eval()
    for ex in split:
        prompt = f"Question: {ex['question']}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        text = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(text)
        gold.append(ex['answer'])
    return accuracy_score(preds, gold)


base_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='auto')
s_base = eval_model(base_model, eval_split, tokenizer)
print('S_base', s_base)

In [ ]:
out_dir = CKPT_ROOT / 'scan_latest'
out_dir.mkdir(parents=True, exist_ok=True)

full_dir = out_dir / 'full'
full_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='auto')
run_grpo_train(
    full_model,
    grpo_train,
    GrpoRunConfig(output_dir=full_dir, learning_rate=LR, num_train_steps=STEPS, train_layer_indices=None),
)
s_full = eval_model(full_model, eval_split, tokenizer)
print('S_full', s_full)

In [ ]:
n_layers = num_transformer_layers(full_model)


def train_and_eval_layer(k: int) -> float:
    layer_dir = out_dir / f'layer_{k}'
    m = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='auto')
    run_grpo_train(
        m,
        grpo_train,
        GrpoRunConfig(
            output_dir=layer_dir,
            learning_rate=LR,
            num_train_steps=STEPS,
            train_layer_indices=[k],
        ),
    )
    return eval_model(m, eval_split, tokenizer)


result = resume_layer_scan(
    out_dir,
    n_layers,
    train_and_eval_layer,
    s_base=s_base,
    s_full=s_full,
    config={'model': MODEL, 'steps': STEPS, 'lr': LR, 'seed': SEED},
)
print(result.out_dir)
print('best C', max(result.contributions.values()))